In [26]:
# ==============================================================================
# LOCKED-MARKET DIP SCANNER — strategy config
# ==============================================================================

# Remove old settlement scanner code/config (safe if they don't exist)
for name in ["SETTLE_CFG", "scan_for_settlement_edges", "find_near_settle_events",
             "record_settlement_trades", "size_settlement_trade", "settle_loop_once",
             "realistic_bounds", "realistic_bounds_band", "implied_prob_at_close",
             "true_prob_for_market", "describe_market", "describe_trade",
             "parse_market_fields_v2", "detect_market_type",
             "start_settlement_scanner", "stop_settlement_scanner", 
             "settle_scanner_status", "dry_run_settlement_scan",
             "run_settlement_scan_now", "_settle_scanner_thread", "_settle_scanner_stop",
             "close_all_settlement_positions", "settlement_report",
             "settlement_portfolio_metrics", "ensure_settle_schema",
             "ensure_description_column"]:
    globals().pop(name, None)

# New config for the lock-and-dip strategy
LOCK_CFG = {
    # Lock detection
    "lock_percentile":        99.0,     # empirical percentile of historical moves
    "historical_days":        90,       # how much BTC history to use for thresholds
    "scan_window_min":        10,       # only scan markets in final N minutes
    "scan_interval_sec":      10,       # scan frequency during window
    
    # Trade triggers
    "yes_lock_fair_value":    0.99,     # what a YES-locked market "should" trade at
    "no_lock_fair_value":     0.01,     # what a NO-locked market's YES price should be
    "min_dip_cents":          3.0,      # adjustable — how far below fair before we buy
    
    # Safety rails
    "max_spread_cents":       8.0,      # skip if spread wider than this
    "min_recent_trades":      2,        # skip if market has no recent activity
    "recent_trade_window_sec": 300,     # "recent" = last 5 min
    "min_open_interest":      50,
    
    # Sizing — small bets, many reps
    "max_dollars_per_trade":  15.00,
    "min_dollars_per_trade":  2.00,
    "max_per_market":         1,        # one position per market at a time
    "max_total_exposure_pct": 0.10,     # 10% of bankroll max
    
    # Fees
    "kalshi_fee_cents":       0.7,
}

print("Old settlement code removed. LOCK_CFG loaded:")
for k, v in LOCK_CFG.items():
    print(f"  {k:<28s} {v}")

Old settlement code removed. LOCK_CFG loaded:
  lock_percentile              99.0
  historical_days              90
  scan_window_min              10
  scan_interval_sec            10
  yes_lock_fair_value          0.99
  no_lock_fair_value           0.01
  min_dip_cents                3.0
  max_spread_cents             8.0
  min_recent_trades            2
  recent_trade_window_sec      300
  min_open_interest            50
  max_dollars_per_trade        15.0
  min_dollars_per_trade        2.0
  max_per_market               1
  max_total_exposure_pct       0.1
  kalshi_fee_cents             0.7


Exception in thread Thread-5:
Traceback (most recent call last):
  File "/Users/rithvikijju/opt/anaconda3/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/Users/rithvikijju/opt/anaconda3/lib/python3.8/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/var/folders/gm/9yrzs9bn5hb5pt_c3q7l8s4c0000gn/T/ipykernel_89273/1932985348.py", line 19, in _settle_scan_worker
NameError: name '_settle_scanner_stop' is not defined


In [27]:
# ==============================================================================
# HISTORICAL MOVE PERCENTILES
# Computes the p-th percentile of absolute BTC moves over N minutes,
# from historical minute bars. This is our "what's reachable" table.
# ==============================================================================

def build_move_percentile_table(bars: pd.DataFrame, minutes_list: list, 
                                   percentile: float) -> dict:
    """For each N in minutes_list, compute the p-th percentile of |BTC move over N min|.
    Returns {minutes: dollar_move_threshold} computed as a fraction of price to stay scale-free."""
    bars = bars.sort_values("time").reset_index(drop=True)
    
    table = {}
    for n in minutes_list:
        # For each possible starting point, compute the move over n minutes
        prices = bars["close"].values
        if len(prices) < n + 1:
            table[n] = np.nan
            continue
        starts = prices[:-n]
        ends = prices[n:]
        abs_pct_moves = np.abs((ends - starts) / starts)
        table[n] = float(np.percentile(abs_pct_moves, percentile))
    return table


def max_realistic_move_fraction(ttl_minutes: float, move_table: dict) -> float:
    """Linear-interpolate from the move percentile table for an arbitrary TTL in minutes.
    Returns a fractional move (e.g., 0.0025 = 0.25%)."""
    if ttl_minutes <= 0:
        return 0.0
    keys = sorted(move_table.keys())
    if ttl_minutes <= keys[0]:
        return move_table[keys[0]]
    if ttl_minutes >= keys[-1]:
        # Extrapolate with sqrt-of-time (standard vol scaling)
        last = keys[-1]
        return move_table[last] * np.sqrt(ttl_minutes / last)
    # Linear interp between the two surrounding keys
    lo = max(k for k in keys if k <= ttl_minutes)
    hi = min(k for k in keys if k >= ttl_minutes)
    if lo == hi:
        return move_table[lo]
    frac = (ttl_minutes - lo) / (hi - lo)
    return move_table[lo] + frac * (move_table[hi] - move_table[lo])


# Build the table once on startup
print(f"Building empirical move table from {LOCK_CFG['historical_days']} days of BTC history...")
# Ensure we have enough data
if len(btc_1m) < LOCK_CFG["historical_days"] * 1440 * 0.8:
    print("  Fetching more history...")
    btc_1m_extra = fetch_historical_minutes(days_back=LOCK_CFG["historical_days"])
    btc_1m_extra["log_ret"] = np.log(btc_1m_extra["close"] / btc_1m_extra["close"].shift(1))
    btc_1m = pd.concat([btc_1m, btc_1m_extra]).drop_duplicates("time").sort_values("time").reset_index(drop=True)

move_table = build_move_percentile_table(
    btc_1m,
    minutes_list=[1, 2, 3, 5, 7, 10, 15, 20, 30],
    percentile=LOCK_CFG["lock_percentile"],
)

print(f"\np{LOCK_CFG['lock_percentile']} absolute move over N minutes:")
for n in sorted(move_table.keys()):
    pct = move_table[n] * 100
    # Also show the approximate dollar move at current spot levels
    dollar_move_at_80k = 80_000 * move_table[n]
    print(f"  {n:>3d} min:  {pct:.3f}%  (~${dollar_move_at_80k:.0f} at $80k BTC)")

print("\nmax_realistic_move_fraction(ttl_min) now available.")

Building empirical move table from 90 days of BTC history...
  Fetching more history...

p99.0 absolute move over N minutes:
    1 min:  0.284%  (~$227 at $80k BTC)
    2 min:  0.396%  (~$317 at $80k BTC)
    3 min:  0.488%  (~$390 at $80k BTC)
    5 min:  0.638%  (~$510 at $80k BTC)
    7 min:  0.749%  (~$599 at $80k BTC)
   10 min:  0.898%  (~$718 at $80k BTC)
   15 min:  1.088%  (~$871 at $80k BTC)
   20 min:  1.241%  (~$993 at $80k BTC)
   30 min:  1.535%  (~$1228 at $80k BTC)

max_realistic_move_fraction(ttl_min) now available.


In [28]:
# ==============================================================================
# LOCK DETECTION
# Given a Kalshi BTC market and current spot + ttl, determine whether it's
# locked-in to YES, locked-in to NO, or still uncertain.
# ==============================================================================

def detect_market_type(ticker: str) -> str:
    """cumulative (KXBTCD-...-T<strike>) or bucket (KXBTC-...-B<floor>)."""
    if "-T" in ticker:
        return "cumulative"
    if "-B" in ticker:
        return "bucket"
    return "unknown"


def parse_market(m: dict) -> dict:
    """Extract what we need from a Kalshi market object."""
    def _f(key, default=None):
        v = m.get(key, default)
        if isinstance(v, str):
            try: return float(v)
            except: return default
        return v
    
    yes_bid = _f("yes_bid_dollars") or _f("yes_bid")
    yes_ask = _f("yes_ask_dollars") or _f("yes_ask")
    if yes_bid and yes_bid > 1: yes_bid /= 100
    if yes_ask and yes_ask > 1: yes_ask /= 100
    
    return {
        "ticker":       m.get("ticker", ""),
        "market_type":  detect_market_type(m.get("ticker", "")),
        "floor":        _f("floor_strike"),
        "cap":          _f("cap_strike"),
        "yes_bid":      yes_bid,
        "yes_ask":      yes_ask,
        "open_interest": _f("open_interest_fp") or _f("open_interest") or 0,
        "volume_24h":   _f("volume_fp") or _f("volume") or 0,
        "close_time":   m.get("close_time"),
        "status":       m.get("status"),
    }


def describe_market(parsed: dict) -> str:
    """Plain-English description of what YES means on this market."""
    mt = parsed["market_type"]
    floor, cap = parsed["floor"], parsed["cap"]
    if mt == "cumulative":
        return f"BTC ≥ ${floor:,.0f}"
    if mt == "bucket":
        if cap:
            return f"BTC between ${floor:,.0f} and ${cap:,.0f}"
        return f"BTC in bucket at ${floor:,.0f}"
    return "unknown market"


def classify_lock(parsed: dict, spot: float, ttl_minutes: float, 
                   move_table: dict) -> dict:
    """Determine whether this market is locked-in YES, NO, or neither.
    
    Returns:
      {
        "locked":      "yes" | "no" | None,
        "distance":    how many thresholds of "reachable" we are from the boundary,
        "reasoning":   string explaining the classification,
      }
    """
    if ttl_minutes <= 0:
        return {"locked": None, "distance": 0, "reasoning": "already closed"}
    
    max_move_frac = max_realistic_move_fraction(ttl_minutes, move_table)
    max_move_dollars = spot * max_move_frac
    
    floor = parsed["floor"]
    cap = parsed["cap"]
    
    if parsed["market_type"] == "cumulative":
        # YES means BTC ≥ floor. 
        # Locked YES: spot - floor > max_reachable_drop
        # Locked NO:  floor - spot > max_reachable_rise
        margin_above = spot - floor
        if margin_above > max_move_dollars:
            return {
                "locked": "yes",
                "distance": margin_above / max_move_dollars,
                "reasoning": f"spot ${spot:,.0f} is ${margin_above:,.0f} above strike; "
                             f"max reachable drop in {ttl_minutes:.1f}m is ${max_move_dollars:,.0f}",
            }
        elif -margin_above > max_move_dollars:
            return {
                "locked": "no",
                "distance": -margin_above / max_move_dollars,
                "reasoning": f"spot ${spot:,.0f} is ${-margin_above:,.0f} below strike; "
                             f"max reachable rise in {ttl_minutes:.1f}m is ${max_move_dollars:,.0f}",
            }
        return {"locked": None, "distance": 0, 
                "reasoning": f"strike ${floor:,.0f} within reach of spot ${spot:,.0f}"}
    
    elif parsed["market_type"] == "bucket":
        # YES means floor ≤ BTC ≤ cap.
        # Locked YES: spot is well inside [floor, cap] AND both edges are > max_move away
        # Locked NO:  spot is outside [floor, cap] by more than max_move
        if cap is None:
            return {"locked": None, "distance": 0, "reasoning": "bucket cap unknown"}
        
        inside = (floor <= spot <= cap)
        if inside:
            dist_to_floor = spot - floor
            dist_to_cap = cap - spot
            min_dist = min(dist_to_floor, dist_to_cap)
            if min_dist > max_move_dollars:
                return {
                    "locked": "yes",
                    "distance": min_dist / max_move_dollars,
                    "reasoning": f"spot ${spot:,.0f} inside [{floor:,.0f}, {cap:,.0f}]; "
                                 f"nearest edge ${min_dist:,.0f} away > ${max_move_dollars:,.0f} max move",
                }
            return {"locked": None, "distance": 0,
                    "reasoning": f"spot inside bucket but close to edge"}
        else:
            dist_to_bucket = min(abs(spot - floor), abs(spot - cap))
            if dist_to_bucket > max_move_dollars:
                return {
                    "locked": "no",
                    "distance": dist_to_bucket / max_move_dollars,
                    "reasoning": f"spot ${spot:,.0f} is ${dist_to_bucket:,.0f} outside bucket; "
                                 f"max reach ${max_move_dollars:,.0f}",
                }
            return {"locked": None, "distance": 0,
                    "reasoning": "spot outside bucket but within reach"}
    
    return {"locked": None, "distance": 0, "reasoning": "unknown market type"}


print("Lock detection loaded: classify_lock(parsed, spot, ttl_min, move_table)")

Lock detection loaded: classify_lock(parsed, spot, ttl_min, move_table)


In [29]:
# ==============================================================================
# SCAN FOR DIPS ON LOCKED MARKETS
# ==============================================================================

def find_near_close_events(window_min: float) -> list:
    """Find BTC events with at least one market closing within `window_min` minutes."""
    now = datetime.now(timezone.utc)
    all_events = []
    for series in ["KXBTCD", "KXBTC"]:
        try:
            resp = kalshi_prod.get_events(series_ticker=series, status="open", limit=50)
            for e in resp.get("events", []):
                e["_series"] = series
                all_events.append(e)
        except Exception:
            continue
    
    near = []
    for e in all_events:
        try:
            mkts = kalshi_prod.get_markets(event_ticker=e["event_ticker"], limit=100).get("markets", [])
            if not mkts:
                continue
            closes = [dtparser.isoparse(m["close_time"]) for m in mkts if m.get("close_time")]
            close = min(closes) if closes else None
            if not close:
                continue
            ttl_min = (close - now).total_seconds() / 60
            if 0 < ttl_min <= window_min:
                near.append({
                    "event_ticker": e["event_ticker"],
                    "series":       e["_series"],
                    "close_time":   close,
                    "ttl_minutes":  ttl_min,
                    "markets":      mkts,
                })
        except Exception:
            continue
    
    near.sort(key=lambda x: x["ttl_minutes"])
    return near


def has_recent_activity(market_ticker: str, window_sec: int, min_trades: int) -> bool:
    """Check if market has had recent trades (to avoid stale quotes)."""
    try:
        resp = kalshi_prod._get("/markets/trades", 
                                 {"ticker": market_ticker, "limit": 20})
        trades = resp.get("trades", [])
        if len(trades) < min_trades:
            return False
        cutoff = datetime.now(timezone.utc) - timedelta(seconds=window_sec)
        recent = [t for t in trades if dtparser.isoparse(t["created_time"]) >= cutoff]
        return len(recent) >= min_trades
    except Exception:
        return True  # if we can't check, don't block


def scan_for_dip_trades(cfg: dict = LOCK_CFG, verbose: bool = True) -> pd.DataFrame:
    """Main scan: find locked-in markets with price dips worth buying."""
    now = datetime.now(timezone.utc)
    spot = get_btc_spot()
    if not spot or spot <= 0:
        return pd.DataFrame()
    
    events = find_near_close_events(cfg["scan_window_min"])
    if verbose:
        print(f"\n[{now.strftime('%H:%M:%S')}] BTC ${spot:,.2f}  |  "
              f"{len(events)} events in {cfg['scan_window_min']}m window")
    
    if not events:
        return pd.DataFrame()
    
    trades = []
    for event in events:
        ttl_min = event["ttl_minutes"]
        
        for m in event["markets"]:
            parsed = parse_market(m)
            if parsed["market_type"] == "unknown":
                continue
            if parsed["floor"] is None or parsed["yes_bid"] is None or parsed["yes_ask"] is None:
                continue
            if parsed["open_interest"] < cfg["min_open_interest"]:
                continue
            
            spread_cents = (parsed["yes_ask"] - parsed["yes_bid"]) * 100
            if spread_cents > cfg["max_spread_cents"]:
                continue
            
            # Is this market locked in?
            lock = classify_lock(parsed, spot, ttl_min, move_table)
            if lock["locked"] is None:
                continue
            
            # YES-locked: fair value ~0.99, check for dip in ask
            if lock["locked"] == "yes":
                fair = cfg["yes_lock_fair_value"]
                ask = parsed["yes_ask"]
                dip_cents = (fair - ask) * 100
                if dip_cents < cfg["min_dip_cents"]:
                    continue
                
                # Check recent activity
                if not has_recent_activity(parsed["ticker"], 
                                             cfg["recent_trade_window_sec"],
                                             cfg["min_recent_trades"]):
                    continue
                
                expected_profit = (fair - ask) - (cfg["kalshi_fee_cents"] / 100)
                trades.append({
                    "event":         event["event_ticker"],
                    "ticker":        parsed["ticker"],
                    "market_type":   parsed["market_type"],
                    "description":   f"BUY YES: {describe_market(parsed)} (locked YES)",
                    "side":          "yes",
                    "entry_price":   ask,
                    "fair_value":    fair,
                    "dip_cents":     dip_cents,
                    "exp_profit":    expected_profit,
                    "ttl_min":       ttl_min,
                    "lock_distance": lock["distance"],
                    "reasoning":     lock["reasoning"],
                    "spot":          spot,
                    "open_interest": parsed["open_interest"],
                })
            
            # NO-locked: YES fair value ~0.01, check if yes_bid is elevated (pay to take NO cheaply)
            elif lock["locked"] == "no":
                fair = cfg["no_lock_fair_value"]
                bid = parsed["yes_bid"]
                dip_cents = (bid - fair) * 100  # how much above fair is the bid
                if dip_cents < cfg["min_dip_cents"]:
                    continue
                
                if not has_recent_activity(parsed["ticker"],
                                             cfg["recent_trade_window_sec"],
                                             cfg["min_recent_trades"]):
                    continue
                
                # Buy NO at (1 - yes_bid). Fair value of NO is (1 - fair).
                no_cost = 1 - bid
                no_fair = 1 - fair
                expected_profit = (no_fair - no_cost) - (cfg["kalshi_fee_cents"] / 100)
                trades.append({
                    "event":         event["event_ticker"],
                    "ticker":        parsed["ticker"],
                    "market_type":   parsed["market_type"],
                    "description":   f"BUY NO: {describe_market(parsed)} (locked NO)",
                    "side":          "no",
                    "entry_price":   no_cost,
                    "fair_value":    no_fair,
                    "dip_cents":     dip_cents,
                    "exp_profit":    expected_profit,
                    "ttl_min":       ttl_min,
                    "lock_distance": lock["distance"],
                    "reasoning":     lock["reasoning"],
                    "spot":          spot,
                    "open_interest": parsed["open_interest"],
                })
    
    df = pd.DataFrame(trades)
    if verbose and len(df) > 0:
        print(f"\n--- {len(df)} dip opportunities found ---")
        for _, t in df.iterrows():
            print(f"\n  → {t['description']}")
            print(f"    entry ${t['entry_price']:.3f} vs fair ${t['fair_value']:.3f}  "
                  f"({t['dip_cents']:+.1f}¢ dip, expected ${t['exp_profit']:.3f}/contract)")
            print(f"    {t['reasoning']}")
            print(f"    lock distance: {t['lock_distance']:.2f}x threshold, "
                  f"{t['ttl_min']:.1f}m to close")
    
    return df


print("Scanner loaded: scan_for_dip_trades(verbose=True)")

Scanner loaded: scan_for_dip_trades(verbose=True)


In [30]:
# ==============================================================================
# SIZE + RECORD + MAIN LOOP
# ==============================================================================

def ensure_lock_schema():
    conn = _db_conn()
    existing = {r[1] for r in conn.execute("PRAGMA table_info(paper_trades)").fetchall()}
    needed = {
        "trade_type":     "TEXT DEFAULT 'single'",
        "description":    "TEXT",
        "market_type":    "TEXT",
        "lock_fair":      "REAL",
        "lock_dip_cents": "REAL",
        "lock_distance":  "REAL",
        "lock_ttl_min":   "REAL",
        "lock_reasoning": "TEXT",
    }
    for col, ctype in needed.items():
        if col not in existing:
            conn.execute(f"ALTER TABLE paper_trades ADD COLUMN {col} {ctype}")
    conn.commit()
    conn.close()

ensure_lock_schema()


def size_dip_trade(trade, cfg: dict = LOCK_CFG) -> tuple:
    """Sizing for a dip trade. Since the edge is small but high-probability,
    we size to fixed dollar amount rather than Kelly (Kelly would oversize on ~95% win bets)."""
    equity, exposure = current_bankroll_and_exposure()
    
    # Fixed-dollar sizing: bet max_dollars_per_trade per opportunity
    budget = min(cfg["max_dollars_per_trade"],
                  equity * cfg["max_total_exposure_pct"] - exposure)
    if budget <= cfg["min_dollars_per_trade"]:
        return 0, 0.0
    
    entry = trade["entry_price"]
    contracts = int(np.floor(budget / entry)) if entry > 0 else 0
    cost = contracts * entry
    if cost < cfg["min_dollars_per_trade"]:
        return 0, 0.0
    return contracts, cost


def record_dip_trades(trades_df, cfg: dict = LOCK_CFG) -> int:
    if len(trades_df) == 0:
        return 0
    
    now_iso = datetime.now(timezone.utc).isoformat()
    cutoff = (datetime.now(timezone.utc) - timedelta(minutes=2)).isoformat()
    conn = _db_conn()
    recorded = 0
    
    for _, t in trades_df.iterrows():
        # Dedup: same ticker+side within 2 min
        existing = conn.execute("""
            SELECT id FROM paper_trades 
            WHERE market_ticker=? AND side=? AND trade_type='lock_dip' 
              AND timestamp_utc >= ? AND settled=0
            LIMIT 1
        """, (t["ticker"], t["side"], cutoff)).fetchone()
        if existing:
            continue
        
        contracts, cost = size_dip_trade(t)
        if contracts == 0:
            continue
        
        conn.execute("""
            INSERT INTO paper_trades (
              timestamp_utc, event_ticker, market_ticker, side,
              entry_price, contracts, entry_edge_cents, model_p_yes, market_yes_mid,
              btc_spot_entry, confidence, trade_type,
              description, market_type, lock_fair, lock_dip_cents, lock_distance,
              lock_ttl_min, lock_reasoning
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            now_iso, t["event"], t["ticker"], t["side"],
            float(t["entry_price"]), int(contracts),
            float(t["dip_cents"]),
            float(t["fair_value"]) if t["side"] == "yes" else float(1 - t["fair_value"] + 0),
            float(t["entry_price"]),
            float(t["spot"]), 1.0, "lock_dip",
            str(t["description"]), str(t["market_type"]),
            float(t["fair_value"]), float(t["dip_cents"]), float(t["lock_distance"]),
            float(t["ttl_min"]), str(t["reasoning"]),
        ))
        recorded += 1
        print(f"  ✓ {t['description']}  |  {contracts}@${t['entry_price']:.3f}  "
              f"({t['dip_cents']:+.1f}¢ dip)")
    
    conn.commit()
    conn.close()
    return recorded


def lock_loop_once(verbose: bool = True):
    """One iteration: settle old, scan for dips, size, record."""
    try:
        n = check_settlements()
        if n > 0 and verbose:
            print(f"Settled {n} trades.")
    except Exception as e:
        print(f"Settlement error: {e}")
    
    trades = scan_for_dip_trades(verbose=verbose)
    if len(trades) > 0:
        record_dip_trades(trades)


print("Main loop loaded: lock_loop_once()")

Main loop loaded: lock_loop_once()


In [31]:
# ==============================================================================
# BACKGROUND SCANNER + REPORTING
# ==============================================================================

_lock_scanner_thread = None
_lock_scanner_stop = threading.Event()
LOCK_LOG_PATH = Path.home() / ".btc_kalshi_bot" / "lock_scanner.log"


def _lock_worker(interval_sec: int):
    import io, contextlib
    with open(LOCK_LOG_PATH, "a") as logf:
        logf.write(f"\n[{datetime.now(timezone.utc).isoformat()}] Lock-dip scanner started "
                   f"(interval {interval_sec}s)\n")
        logf.flush()
        while not _lock_scanner_stop.is_set():
            try:
                buf = io.StringIO()
                with contextlib.redirect_stdout(buf):
                    lock_loop_once(verbose=True)
                out = buf.getvalue().strip()
                if out:
                    logf.write(f"\n[{datetime.now(timezone.utc).strftime('%H:%M:%S')}]\n{out}\n")
                    logf.flush()
            except Exception as e:
                logf.write(f"[ERROR] {e}\n")
                logf.flush()
            _lock_scanner_stop.wait(interval_sec)


def start_lock_scanner(interval_sec: int = None):
    global _lock_scanner_thread
    if _lock_scanner_thread and _lock_scanner_thread.is_alive():
        print("Lock scanner already running.")
        return
    interval = interval_sec or LOCK_CFG["scan_interval_sec"]
    _lock_scanner_stop.clear()
    _lock_scanner_thread = threading.Thread(
        target=_lock_worker, args=(interval,), daemon=True)
    _lock_scanner_thread.start()
    print(f"Lock scanner started @ {interval}s interval.  Log: {LOCK_LOG_PATH}")


def stop_lock_scanner():
    global _lock_scanner_thread
    _lock_scanner_stop.set()
    if _lock_scanner_thread:
        _lock_scanner_thread.join(timeout=5)
        _lock_scanner_thread = None
    print("Lock scanner stopped.")


def lock_scanner_status(last_n_lines: int = 30):
    alive = _lock_scanner_thread and _lock_scanner_thread.is_alive()
    print(f"Lock scanner: {'ALIVE' if alive else 'NOT RUNNING'}")
    if LOCK_LOG_PATH.exists():
        with open(LOCK_LOG_PATH) as f:
            lines = f.readlines()
        print(f"Log: {LOCK_LOG_PATH} ({len(lines)} lines)")
        print("\n--- Recent ---")
        for line in lines[-last_n_lines:]:
            print(line.rstrip())


def lock_report(starting_bankroll: float = 1000.0, show_plots: bool = True):
    """Full portfolio metrics for lock_dip trades."""
    conn = _db_conn()
    df = pd.read_sql_query(
        "SELECT * FROM paper_trades WHERE trade_type='lock_dip' ORDER BY timestamp_utc", conn)
    conn.close()
    
    if len(df) == 0:
        print("No lock-dip trades yet.")
        return None
    
    settled = df[df["settled"] == 1].copy()
    open_t  = df[df["settled"] == 0].copy()
    
    realized = settled["pnl_dollars"].sum() if len(settled) else 0.0
    open_val = (open_t["entry_price"] * open_t["contracts"]).sum() if len(open_t) else 0.0
    equity = starting_bankroll + realized
    
    print(f"\n{'='*72}")
    print(f"  LOCK-DIP STRATEGY PORTFOLIO  —  bankroll ${starting_bankroll:.2f}")
    print(f"  {datetime.now(timezone.utc).isoformat()}")
    print(f"{'='*72}")
    print(f"\n┌─ OVERVIEW")
    print(f"│  Equity:              ${equity:>10,.2f}  ({(equity/starting_bankroll-1)*100:+.2f}%)")
    print(f"│  Realized P&L:        ${realized:>+10.2f}")
    print(f"│  Open exposure:       ${open_val:>10,.2f}  ({len(open_t)} positions)")
    
    if len(settled) > 0:
        wins = (settled["pnl_dollars"] > 0).sum()
        n = len(settled)
        print(f"\n┌─ SETTLED")
        print(f"│  Trades:              {n}")
        print(f"│  Wins / Losses:       {wins} / {n - wins}  ({wins/n*100:.1f}%)")
        print(f"│  Total P&L:           ${settled['pnl_dollars'].sum():+.2f}")
        print(f"│  Avg win:             ${settled.loc[settled['pnl_dollars']>0,'pnl_dollars'].mean() if wins else 0:+.3f}")
        print(f"│  Avg loss:            ${settled.loc[settled['pnl_dollars']<=0,'pnl_dollars'].mean() if (n-wins) else 0:+.3f}")
        print(f"│  Avg dip captured:    {settled['lock_dip_cents'].mean():.2f}¢")
        print(f"│  Realized edge:       {(settled['pnl_dollars']/settled['contracts']*100).mean():+.2f}¢")
        
        # By lock_distance bucket
        settled["dist_bucket"] = pd.cut(settled["lock_distance"],
                                          bins=[0, 1.2, 1.5, 2.0, 5, 100],
                                          labels=["1.0-1.2x", "1.2-1.5x", "1.5-2x", "2-5x", ">5x"])
        by_dist = settled.groupby("dist_bucket", observed=True).agg(
            n=("id", "count"),
            wins=("pnl_dollars", lambda x: (x > 0).sum()),
            total=("pnl_dollars", "sum"),
        ).reset_index()
        by_dist["hit"] = (by_dist["wins"] / by_dist["n"] * 100).round(1)
        print(f"\n┌─ BY LOCK DISTANCE (margin beyond p99 move)")
        print(by_dist.to_string(index=False))
        
        by_side = settled.groupby("side").agg(
            n=("id", "count"),
            wins=("pnl_dollars", lambda x: (x > 0).sum()),
            total=("pnl_dollars", "sum"),
        ).reset_index()
        by_side["hit"] = (by_side["wins"] / by_side["n"] * 100).round(1)
        print(f"\n┌─ BY SIDE")
        print(by_side.to_string(index=False))
    
    if len(open_t) > 0:
        print(f"\n┌─ OPEN ({len(open_t)})")
        cols = ["timestamp_utc", "description", "entry_price", "contracts", 
                "lock_dip_cents", "lock_ttl_min"]
        print(open_t[cols].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    
    if show_plots and len(settled) >= 3:
        settled["timestamp_utc"] = pd.to_datetime(settled["timestamp_utc"])
        settled = settled.sort_values("timestamp_utc")
        settled["cum_pnl"] = settled["pnl_dollars"].cumsum()
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(settled["timestamp_utc"], starting_bankroll + settled["cum_pnl"], marker="o")
        axes[0].axhline(starting_bankroll, color="k", ls="--", alpha=0.4)
        axes[0].set_title("Equity"); axes[0].grid(alpha=0.3); axes[0].tick_params(axis='x', rotation=30)
        axes[1].hist(settled["pnl_dollars"], bins=15, edgecolor="black", alpha=0.7)
        axes[1].axvline(0, color="k", ls="--"); axes[1].set_title("PnL distribution"); axes[1].grid(alpha=0.3)
        axes[2].scatter(settled["lock_dip_cents"], settled["pnl_dollars"],
                        c=settled["pnl_dollars"].apply(lambda x: "green" if x > 0 else "red"), alpha=0.6)
        axes[2].set_xlabel("Dip cents at entry"); axes[2].set_ylabel("PnL")
        axes[2].set_title("PnL vs dip size"); axes[2].grid(alpha=0.3)
        plt.tight_layout(); plt.show()
    
    return df


print("\n" + "="*70)
print("LOCK-DIP SCANNER FULLY LOADED")
print("="*70)
print("""
Manual:
  scan_for_dip_trades()           single scan, no trades recorded
  lock_loop_once()                one full iteration (settle + scan + record)

Background:
  start_lock_scanner()            start 10s loop
  stop_lock_scanner()
  lock_scanner_status()

Reporting:
  lock_report()                   full metrics + plots

Tuning:
  LOCK_CFG["min_dip_cents"] = 4.0   adjust anytime
  LOCK_CFG["lock_percentile"] = 99.5   adjust (requires rebuilding move_table)
""")


LOCK-DIP SCANNER FULLY LOADED

Manual:
  scan_for_dip_trades()           single scan, no trades recorded
  lock_loop_once()                one full iteration (settle + scan + record)

Background:
  start_lock_scanner()            start 10s loop
  stop_lock_scanner()
  lock_scanner_status()

Reporting:
  lock_report()                   full metrics + plots

Tuning:
  LOCK_CFG["min_dip_cents"] = 4.0   adjust anytime
  LOCK_CFG["lock_percentile"] = 99.5   adjust (requires rebuilding move_table)



In [37]:
stop_lock_scanner()

Lock scanner stopped.


In [36]:
lock_scanner_status()

Lock scanner: ALIVE
Log: /Users/rithvikijju/.btc_kalshi_bot/lock_scanner.log (8 lines)

--- Recent ---

[2026-04-22T17:16:59.762466+00:00] Lock-dip scanner started (interval 10s)

[17:17:00]
[17:16:59] BTC $78,925.26  |  0 events in 10m window

[17:17:11]
[17:17:10] BTC $78,925.26  |  0 events in 10m window
